# 01 · Provider landscape

Three ways to integrate a model, from most to least portable:

| Approach | Example | Trade-off |
|---|---|---|
| OpenAI-compatible endpoint | `toolkit.llm` (OpenAI, Anthropic, Gemini, Groq, Ollama...) | one client; provider-specific features partly unavailable |
| Abstraction library | LiteLLM | one call signature, normalized usage/cost reporting |
| Router service | OpenRouter | one key and bill for hundreds of models; extra hop |
| Native SDK | `anthropic`, `google-genai` | full feature access (caching controls, content blocks), provider lock-in |

Also covered: inference-time reasoning effort, and prompt caching economics.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, for `toolkit`

In [ ]:
import time

from toolkit import available_providers, completion

available_providers()

## Native SDKs

The Anthropic Messages API differs from Chat Completions in a few ways that matter when porting:
`system` is a top-level parameter rather than a message, `max_tokens` is required,
and responses are lists of typed content blocks.

In [ ]:
from anthropic import Anthropic

prompt = "Describe the colour blue to someone who has never seen, in one sentence."

try:
    msg = Anthropic().messages.create(
        model="claude-haiku-5-5",
        system="You are a poet.",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}],
    )
    print(msg.content[0].text, msg.usage, sep="\n")
except Exception as exc:
    print("skipped:", exc)

In [ ]:
from google import genai

try:
    print(
        genai.Client().models.generate_content(model="gemini-3.1-flash-lite", contents=prompt).text
    )
except Exception as exc:
    print("skipped:", exc)

## LiteLLM: one signature, normalized usage and cost

Model strings are `provider/model`. Every response carries token usage, and LiteLLM attaches
a cost estimate from its pricing table, which is handy for budgeting experiments.

In [ ]:
from litellm import completion as litellm_completion


def report(response):
    u = response.usage
    cost = (response._hidden_params.get("response_cost") or 0) * 100
    cached = getattr(u.prompt_tokens_details, "cached_tokens", 0) if u.prompt_tokens_details else 0
    print(f"in={u.prompt_tokens} (cached {cached})  out={u.completion_tokens}  cost={cost:.4f}¢")


r = litellm_completion(
    model="openai/gpt-6-luna",
    messages=[{"role": "user", "content": "Name three uses of a Bloom filter. One line each."}],
)
print(r.choices[0].message.content)
report(r)

## Inference-time reasoning

Reasoning models can spend extra tokens "thinking" before answering. `reasoning_effort` trades
latency and cost for accuracy. Classic trap question below: the correct answer is 2/3, not 1/2.

In [ ]:
puzzle = [
    {
        "role": "user",
        "content": "I toss two fair coins. At least one is heads. "
        "What's the probability the other is tails? Answer with the probability only.",
    }
]

for model, effort in [("gpt-6-luna", "none"), ("gpt-6-luna", "low"), ("gpt-6.1-sol", "low")]:
    start = time.perf_counter()
    r = completion(puzzle, provider="openai", model=model, reasoning_effort=effort)
    details = r.usage.completion_tokens_details
    reasoning = details.reasoning_tokens if details else 0
    print(
        f"{model:12} effort={effort:5} -> {r.choices[0].message.content!r:8} "
        f"reasoning_tokens={reasoning:4}  {time.perf_counter() - start:.1f}s"
    )

## Prompt caching

Providers cache the processed prefix of a prompt. Repeating a long, identical prefix (system prompt,
reference document, few-shot examples) is billed at a steep discount on cache hits.

| Provider | Mechanism | Pricing |
|---|---|---|
| OpenAI | automatic for prefixes ≥ 1,024 tokens | cached input heavily discounted |
| Anthropic | explicit `cache_control` breakpoints | cache writes cost more, reads ~10% of base |
| Gemini | implicit, plus explicit cache objects | discounted cached tokens; explicit caches billed per hour |

Design rule: **static content first, variable content last**. Any change early in the prompt invalidates everything after it.

In [ ]:
import requests

DATA = Path("data")
DATA.mkdir(exist_ok=True)
book_path = DATA / "art_of_war.txt"
if not book_path.exists():  # public domain, Project Gutenberg #132
    book_path.write_text(
        requests.get("https://www.gutenberg.org/cache/epub/132/pg132.txt", timeout=30).text
    )

book = book_path.read_text()
print(f"{len(book):,} chars")

In [ ]:
def ask_about_book(question):
    # Static document first, question last, so the prefix is identical across calls
    messages = [
        {"role": "system", "content": "Answer questions using only this text:\n\n" + book},
        {"role": "user", "content": question},
    ]
    r = litellm_completion(model="openai/gpt-6-luna", messages=messages)
    print(r.choices[0].message.content[:300])
    report(r)


ask_about_book("What does Sun Tzu say about attacking by fire? Two sentences.")

In [ ]:
# Same prefix, different question: expect a large cached_tokens count and lower cost
ask_about_book("What are the five factors that govern the art of war? List them.")